In [1]:
using LowLevelFEM

In [2]:
gmsh.initialize()
gmsh.open("truss.geo")

Info    : Reading 'truss.geo'...
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 60%] Meshing curve 2 (Line)
Info    : Done meshing 1D (Wall 0.000168421s, CPU 0.00017s)
Info    : 3 nodes 5 elements
Info    : Done reading 'truss.geo'


In [3]:
mat1 = Material("L1", A=100)
mat2 = Material("L2", A=200)
prob = Problem([mat1, mat2], type=:Truss);

In [4]:
bc1 = BoundaryCondition("P1", ux=0, uy=0)
bc2 = BoundaryCondition("P2", ux=0, uy=0)

bcall = BoundaryCondition("all", uz=0)

load = BoundaryCondition("P3", fy=-100);

In [5]:
free = freeDoFs(prob, [bcall])
cond = freeDoFs(prob, [bc1, bc2, bcall])

2-element Vector{UInt64}:
 0x0000000000000007
 0x0000000000000008

In [6]:
K = stiffnessMatrix(prob)
K[free, free]

6×6 SparseArrays.SparseMatrixCSC{Float64, Int64} with 19 stored entries:
  10000.0  -10000.0         ⋅     ⋅        ⋅         ⋅ 
 -10000.0   24310.8   -14310.8    ⋅    7155.42  -7155.42
       ⋅   -14310.8    14310.8    ⋅   -7155.42   7155.42
       ⋅         ⋅          ⋅     ⋅        ⋅         ⋅ 
       ⋅     7155.42   -7155.42   ⋅    3577.71  -3577.71
       ⋅    -7155.42    7155.42   ⋅   -3577.71   3577.71

In [7]:
K[cond, cond]

2×2 SparseArrays.SparseMatrixCSC{Float64, Int64} with 4 stored entries:
 24310.8   7155.42
  7155.42  3577.71

In [8]:
f = loadVector(prob, [load])
DoFs(f)[free]

6-element Vector{Float64}:
    0.0
    0.0
    0.0
    0.0
 -100.0
    0.0

In [9]:
showDoFResults(f, name="F")

1

In [10]:
DoFs(f)[cond]

2-element Vector{Float64}:
    0.0
 -100.0

In [11]:
q = solveDisplacement(K, f, support=[bc1, bc2, bcall])
DoFs(q)[free]

6-element Vector{Float64}:
  0.0
  0.019999999999999997
  0.0
  0.0
 -0.06795084971874736
  0.0

In [12]:
showDoFResults(q, name="q", visible=true)

2

In [13]:
N = solveAxialForce(q)
N.A

2-element Vector{Matrix{Float64}}:
 [199.99999999999997; 199.99999999999997;;]
 [-223.60679774997894; -223.60679774997894;;]

In [14]:
showElementResults(N, name="N")

3

In [15]:
fT = K * q - f
DoFs(fT)[free]

6-element Vector{Float64}:
 -199.99999999999997
    0.0
  199.99999999999994
    0.0
    2.842170943040401e-14
   99.99999999999997

In [16]:
DoFs(fT)[constrainedDoFs(prob, [bc1])]

2-element Vector{Float64}:
 -199.99999999999997
    0.0

In [17]:
DoFs(fT)[constrainedDoFs(prob, [bc2])]

2-element Vector{Float64}:
 199.99999999999994
  99.99999999999997

In [ ]:
openPostProcessor()

In [ ]:
gmsh.finalize()